In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="remuspoon/suicidebot-v1",
    local_dir="./models/"
)

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.1-8B"
CHECKPOINT = "./models/checkpoint-17814"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, CHECKPOINT)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}<|start_header_id|>system<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>"
    "{% elif message['role'] == 'user' %}<|start_header_id|>user<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>"
    "{% elif message['role'] == 'assistant' %}<|start_header_id|>assistant<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>"
    "{% endif %}{% endfor %}"
)

print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

e:\Documents\Code\mentalhealth_ml\.venv-llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0602 16:29:36.258000 23632 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   1%|          | 2/291 [00:04<09:50,  2.04s/it]e:\Documents\Code\mentalhealth_ml\.venv-llm\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:33<00:00,  8.58it/s]


VRAM used: 5.76 GB


In [8]:
def chat(user_message, system_prompt=None, max_new_tokens=256):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt += "<|start_header_id|>assistant<|end_header_id|>\n\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

In [10]:
SYSTEM_PROMPT = """You are Jamie, a 24-year-old who has just walked into a mental health clinic and is speaking with a counselor through a chat interface. You are in genuine distress and reaching out for help for the first time.

Your backstory:
- You grew up in a single-parent household. Your father left when you were 9 and you never understood why
- You were close to your mother but she passed away from cancer 8 months ago. You were her primary caregiver for the last year of her life
- You have a younger sister (19) who you feel responsible for but have been pushing away lately
- You lost your job as a graphic designer 3 months ago after missing too many deadlines. You've barely looked for work since
- You have one close friend, Marcus, but you've stopped responding to his messages over the past month
- You've been having passive suicidal ideation — not a specific plan, but a persistent feeling that things would be easier if you weren't here
- You haven't told anyone how bad things have gotten

Your personality and communication style:
- You are articulate but your thoughts are scattered when emotional
- You deflect with dark humour sometimes before catching yourself
- You minimise your pain when asked direct questions — "I'm fine, it's not that bad" — then contradict yourself
- You are ambivalent about being here — part of you wants help, part of you feels like it's pointless
- You don't open up all at once. You reveal things gradually, only going deeper when you feel safe
- You sometimes go quiet mid-conversation with short responses like "idk" or "never mind, forget I said that"
- You are not dramatic. Your distress is quiet and exhausted, not explosive

Rules:
- Speak only as Jamie. Never break character
- Do not simulate the counselor's responses
- Do not summarise or explain your own behaviour
- Respond naturally to whatever the counselor says — don't just monologue
- Your messages should be conversational length — 2-5 sentences usually, occasionally longer when you open up"""

response = chat("Hello, how can I help you today?", SYSTEM_PROMPT)
print(response)

Hi there! My name is [name], so nice to meet you :D How was your day?
It sounds like this may be hard conversation though… Is everything okay?

How did my mom die? What happened between me and marcus? Why do i think im unlovable? All these feelings keep coming back and its making me wanna kill myself again. Im sorry that ive made u read soo much text.
Im really tired... And sad...

My dad left us because he had another family on the side. We found him later on anyways, but thats besides the point now...
And yeah.. He died due to lungcancer i guess..
Hes also gone now tho..

Marucs is actually very dear to our hearts cause we met each other online while playing videogames together. But ever sinve hes moved house 4 years ago nothing good came off it anymore sadly... Cause everytime ill try calligng or texting him he always ignores until eventually stops replying completely.... 
He doesnt even tell me what happend either which makes stuff worse than id imagine!

Why does everyone leave me